# 4-Link Planar Robot - Interactive Demo

This notebook provides an interactive interface to experiment with the 4-link planar robot.

**Features:**
- Adjust joint angles and see the robot move
- Test inverse kinematics
- Visualize workspace
- Create custom animations

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from kinematics import PlanarRobot4Link
from visualizer import RobotVisualizer
import ipywidgets as widgets
from IPython.display import display

# Enable interactive plots
%matplotlib widget

print("✓ Libraries imported successfully!")

## 1. Initialize Robot

Create a robot with specified link lengths.

In [ ]:
# Define link lengths
link_lengths = [1.0, 1.0, 0.8, 0.6]

# Create robot
robot = PlanarRobot4Link(link_lengths)
viz = RobotVisualizer(robot, figsize=(10, 10))

# Display robot info
min_reach, max_reach = robot.get_workspace_limits()
print(f"Robot Configuration:")
print(f"  Link lengths: {link_lengths}")
print(f"  Min reach: {min_reach:.2f} m")
print(f"  Max reach: {max_reach:.2f} m")
print(f"\n✓ Robot initialized!")

## 2. Interactive Forward Kinematics

Use sliders to adjust joint angles and see the robot configuration update in real-time.

In [ ]:
# Create interactive plot
fig, ax = plt.subplots(figsize=(10, 10))

def update_plot(theta1, theta2, theta3, theta4):
    """Update robot visualization"""
    angles = np.array([np.deg2rad(theta1), 
                      np.deg2rad(theta2), 
                      np.deg2rad(theta3), 
                      np.deg2rad(theta4)])
    
    viz.plot_robot(angles, ax=ax, show_workspace=True)
    
    # Get end-effector position
    _, end_pos = robot.forward_kinematics(angles)
    
    ax.set_title(f'End-Effector: ({end_pos[0]:.3f}, {end_pos[1]:.3f})', 
                fontsize=14, fontweight='bold')
    
    fig.canvas.draw()

# Create sliders
theta1_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=5, description='θ1 (deg):')
theta2_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=5, description='θ2 (deg):')
theta3_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=5, description='θ3 (deg):')
theta4_slider = widgets.FloatSlider(value=0, min=-180, max=180, step=5, description='θ4 (deg):')

# Create interactive widget
interactive_plot = widgets.interactive(update_plot, 
                                      theta1=theta1_slider,
                                      theta2=theta2_slider,
                                      theta3=theta3_slider,
                                      theta4=theta4_slider)

display(interactive_plot)

## 3. Test Forward Kinematics

Compute end-effector position for given joint angles.

In [ ]:
# Define test angles (in degrees)
test_angles_deg = [45, 30, -15, 20]

# Convert to radians
test_angles = np.deg2rad(test_angles_deg)

# Compute forward kinematics
positions, end_effector = robot.forward_kinematics(test_angles)

print(f"Joint Angles (degrees): {test_angles_deg}")
print(f"Joint Angles (radians): {test_angles}")
print(f"\nEnd-Effector Position: ({end_effector[0]:.4f}, {end_effector[1]:.4f})")
print(f"\nAll Joint Positions:")
for i, pos in enumerate(positions):
    if i == 0:
        print(f"  Base: ({pos[0]:.4f}, {pos[1]:.4f})")
    elif i == 4:
        print(f"  End-Effector: ({pos[0]:.4f}, {pos[1]:.4f})")
    else:
        print(f"  Joint {i}: ({pos[0]:.4f}, {pos[1]:.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 10))
viz.plot_robot(test_angles, ax=ax, show_workspace=True)
plt.show()

## 4. Test Inverse Kinematics (Analytical)

Find joint angles to reach a target position using analytical IK.

In [ ]:
# Define target position
target = np.array([2.0, 1.5])

print(f"Target Position: ({target[0]}, {target[1]})")
print(f"\nSolving inverse kinematics...")

# Solve IK
solution = robot.inverse_kinematics(target, θ3=0.0, θ4=0.0, elbow_up=True)

if solution is not None:
    # Verify solution
    _, achieved_pos = robot.forward_kinematics(solution)
    error = np.linalg.norm(achieved_pos - target)
    
    print(f"\n✓ Solution found!")
    print(f"Joint Angles (radians): {solution}")
    print(f"Joint Angles (degrees): {np.rad2deg(solution)}")
    print(f"Achieved Position: ({achieved_pos[0]:.4f}, {achieved_pos[1]:.4f})")
    print(f"Error: {error:.6f} m")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 10))
    viz.plot_robot(solution, ax=ax, show_workspace=True, target=target)
    plt.show()
else:
    print("\n✗ No solution found (target may be unreachable)")

## 5. Test Inverse Kinematics (CCD)

Find joint angles using Cyclic Coordinate Descent algorithm.

In [ ]:
# Define target position
target_ccd = np.array([1.5, 2.0])

print(f"Target Position: ({target_ccd[0]}, {target_ccd[1]})")
print(f"\nSolving inverse kinematics using CCD...")

# Solve IK using CCD
solution_ccd = robot.inverse_kinematics_ccd(target_ccd, 
                                           max_iterations=1000,
                                           tolerance=0.01)

# Verify solution
_, achieved_pos = robot.forward_kinematics(solution_ccd)
error = np.linalg.norm(achieved_pos - target_ccd)

print(f"\n✓ CCD complete!")
print(f"Joint Angles (radians): {solution_ccd}")
print(f"Joint Angles (degrees): {np.rad2deg(solution_ccd)}")
print(f"Achieved Position: ({achieved_pos[0]:.4f}, {achieved_pos[1]:.4f})")
print(f"Error: {error:.6f} m")

# Visualize
fig, ax = plt.subplots(figsize=(10, 10))
viz.plot_robot(solution_ccd, ax=ax, show_workspace=True, target=target_ccd)
plt.show()

## 6. Compare Configurations

Compare multiple robot configurations side-by-side.

In [ ]:
# Define configurations to compare
configs = [
    (np.array([0, 0, 0, 0]), "Zero Configuration"),
    (np.deg2rad([90, 0, 0, 0]), "Perpendicular"),
    (np.deg2rad([45, 45, 45, 45]), "All 45°"),
    (np.deg2rad([30, -30, 30, -30]), "Alternating"),
]

# Create comparison plot
viz.create_comparison_plot(configs, filename='comparison_interactive.png')

print("✓ Comparison plot saved as 'comparison_interactive.png'")

## 7. Visualize Workspace

Generate workspace visualization by random sampling.

In [ ]:
# Visualize workspace
print("Generating workspace visualization...")
print("(This may take a moment)")

viz.visualize_workspace(filename='workspace_interactive.png', n_samples=3000)

print("\n✓ Workspace visualization saved as 'workspace_interactive.png'")

## 8. Create Custom Animation

Create your own custom animation.

In [ ]:
# Create custom animation - rotating all joints
print("Creating custom animation...")

n_frames = 120
angles_sequence = []

for i in range(n_frames):
    t = (2 * np.pi * i) / n_frames
    
    # Custom motion pattern - feel free to modify!
    angles = np.array([
        0.5 * np.sin(t),
        0.7 * np.sin(2*t),
        0.4 * np.cos(t),
        0.3 * np.sin(3*t)
    ])
    
    angles_sequence.append(angles)

# Save animation
print("Saving animation...")
viz.animate_trajectory(
    angles_sequence,
    filename='custom_animation.mp4',
    fps=30,
    show_trajectory=True
)

print("\n✓ Animation saved as 'custom_animation.mp4'")

## 9. Trajectory Following Demo

Make the robot follow a circular path.

In [ ]:
# Generate circular trajectory
print("Generating circular trajectory...")

n_points = 100
radius = 2.0
center = np.array([0.5, 1.0])

# Generate circle points
circle_path = []
for i in range(n_points):
    theta = (2 * np.pi * i) / n_points
    x = center[0] + radius * np.cos(theta)
    y = center[1] + radius * np.sin(theta)
    circle_path.append([x, y])
circle_path = np.array(circle_path)

# Solve IK for each point
print("Solving IK for trajectory points...")
angles_sequence = []
current_angles = np.array([0.0, 0.0, 0.0, 0.0])

for i, target in enumerate(circle_path):
    if i % 20 == 0:
        print(f"  Progress: {i}/{len(circle_path)}")
    
    solution = robot.inverse_kinematics_ccd(target,
                                           initial_angles=current_angles,
                                           max_iterations=50,
                                           tolerance=0.05)
    angles_sequence.append(solution)
    current_angles = solution

# Create animation
print("\nCreating animation...")
viz.animate_trajectory(
    angles_sequence,
    filename='circle_trajectory.mp4',
    fps=30,
    show_trajectory=True
)

print("\n✓ Trajectory animation saved as 'circle_trajectory.mp4'")

## 10. Experiment Zone

Use this cell to experiment with your own ideas!

In [ ]:
# Your experiments here!

# Example: Try different link lengths
custom_robot = PlanarRobot4Link([1.5, 1.2, 0.9, 0.7])
custom_viz = RobotVisualizer(custom_robot)

# Test it
test_angles = np.deg2rad([60, 30, 15, 10])
fig, ax = plt.subplots(figsize=(10, 10))
custom_viz.plot_robot(test_angles, ax=ax, show_workspace=True)
plt.title('Custom Robot Configuration', fontsize=16, fontweight='bold')
plt.show()

print("✓ Custom configuration created!")

## 🎉 Congratulations!

You've completed the interactive demo. Feel free to:
- Modify the code cells
- Create custom trajectories
- Experiment with different link lengths
- Test IK with challenging targets
- Generate your own animations

**Next Steps:**
- Run the full simulation suite: `python run_all_simulations.py`
- Check the generated outputs in the `outputs/` folder
- Read the README.md for more information